# 🚀 Bluestone Indic-TTS Distillation & ONNX Runtime Pipeline
### Distill AI4Bharat Indic-TTS into an Ultra-Compact (<30MB) ONNX Model for Android, Node.js & Python

This notebook trains and distills an AI4Bharat Indic-TTS teacher into an ultra-fast, CPU-optimized student model (`StudentVITS`), exports it to ONNX, quantizes it to INT8, and verifies real-time playback.

## 1. Check GPU Environment
Verify Google Colab GPU (T4 / V100 / A100).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/docwiser/bluestone-tts-onnx.git || true
%cd bluestone-tts-onnx
!pip install -r requirements-colab.txt

## 3. Choose Target Indic Language
Starting with Hindi (`hi`). The same process can be chosen for Tamil (`ta`), Telugu (`te`), Marathi (`mr`), Bengali (`bn`), etc.

In [ ]:
LANG = "hi"  # @param ["hi", "ta", "te", "bn", "mr", "gu", "kn", "ml"]
print(f"Selected Language: {LANG}")

# Ensure vocab exists
from frontend.indic_tokenizer import IndicTokenizer
tokenizer = IndicTokenizer(lang=LANG)
tokenizer.save_vocab(f"frontend/vocab_{LANG}.json")
print(f"Vocab size: {tokenizer.vocab_size} tokens")

## 4. Extract Teacher Dataset / Pseudo-Distillation Data
Extract synthetic speech, mel-spectrograms, and alignments from AI4Bharat Teacher.

In [ ]:
DATA_DIR = f"./data/distilled_{LANG}"
!python training/teacher_extractor.py \
    --output_dir {DATA_DIR} \
    --lang {LANG} \
    --teacher_model "ai4bharat/vits_rasa_13"

## 5. Train Student VITS with Distillation Losses
Trains with Mel L1 loss, KL divergence, Duration loss, and Multi-Period/Scale GAN discriminators.

In [ ]:
CONFIG_FILE = f"configs/config_{LANG}.json"
MANIFEST_FILE = f"{DATA_DIR}/manifest.txt"
OUTPUT_DIR = f"./outputs/{LANG}"

!python training/train_distill.py \
    --config {CONFIG_FILE} \
    --manifest {MANIFEST_FILE} \
    --output_dir {OUTPUT_DIR} \
    --epochs 50

## 6. Export PyTorch Student Model to Single-Pass ONNX Graph

In [ ]:
CHECKPOINT = f"{OUTPUT_DIR}/checkpoints/best_student.pt"
ONNX_EXPORT_PATH = f"exported/bluestone_tts_{LANG}.onnx"

!python export/export_onnx.py \
    --checkpoint {CHECKPOINT} \
    --config {CONFIG_FILE} \
    --output {ONNX_EXPORT_PATH}

## 7. Dynamic INT8 Quantization (< 30 MB Model!)
Quantizes linear/conv weights to INT8, slashing size from ~45MB to ~22MB.

In [ ]:
QUANT_ONNX_PATH = f"exported/bluestone_tts_{LANG}_quant.onnx"

!python export/optimize_onnx.py \
    --input {ONNX_EXPORT_PATH} \
    --output {QUANT_ONNX_PATH}

## 8. Benchmark CPU Latency and Real-Time Factor (RTF)

In [ ]:
!python export/benchmark.py \
    --model {QUANT_ONNX_PATH} \
    --vocab frontend/vocab.json \
    --runs 5

## 9. Audio Synthesis & In-Browser Playback

In [ ]:
from IPython.display import Audio, display
from inference.python.infer import BluestoneTTS

engine = BluestoneTTS(model_path=QUANT_ONNX_PATH, vocab_path="frontend/vocab.json")
text = "नमस्ते! ब्लूस्टोन ऑनएक्स टेक्स्ट टू स्पीच इंजन अब एंड्रॉइड और वेब के लिए तैयार है।"
audio = engine.synthesize(text, speed=1.0)
engine.save_wav(audio, "colab_sample.wav")

display(Audio("colab_sample.wav", rate=22050, autoplay=True))

## 10. Package & Download Artifacts
Download the deployable model files for Android, Node.js, and Python.

In [ ]:
!zip -r bluestone_tts_hi_release.zip exported/ frontend/vocab.json configs/config_hi.json
from google.colab import files
files.download("bluestone_tts_hi_release.zip")